In [14]:
import pandas as pd
import os
from datetime import datetime
from pathlib import Path

# 兼容 notebook 启动目录不在项目根目录的情况
candidate_root = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "dataset/ml-1m/movies.dat").exists():
        candidate_root = p
        break

if candidate_root is None:
    raise FileNotFoundError("未找到 dataset/ml-1m/movies.dat，请确认数据集路径")

DATA_DIR = candidate_root / "dataset/ml-1m"
movies = pd.read_csv(
        os.path.join(DATA_DIR, "movies.dat"),
        sep="::",
        engine="python",
        names=["MovieID", "Title", "Genres"],
        encoding="latin-1"
    )
    
print(f"总电影数量: {len(movies):,}")
print(f"\n电影数据前10行:")
print(movies.head(10))

# 分析电影类型
all_genres = []
for genres in movies["Genres"]:
    if genres and genres != "(no genres listed)":
        all_genres.extend(genres.split("|"))

genre_counts = pd.Series(all_genres).value_counts()
print(len(genre_counts))
print(f"\n电影类型统计 (前10):")
print(genre_counts.head(10))

总电影数量: 3,883

电影数据前10行:
   MovieID                               Title                        Genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy
5        6                         Heat (1995)         Action|Crime|Thriller
6        7                      Sabrina (1995)                Comedy|Romance
7        8                 Tom and Huck (1995)          Adventure|Children's
8        9                 Sudden Death (1995)                        Action
9       10                    GoldenEye (1995)     Action|Adventure|Thriller
18

电影类型统计 (前10):
Drama         1603
Comedy        1200
Action         503
Thriller       492
Romance        471
Horror         3

In [15]:
from gensim.models import Word2Vec

# 将每部电影的 Genres 拆成词列表，作为 Word2Vec 的训练语料
sentences = [g.split("|") for g in movies["Genres"].dropna().astype(str)]

# 训练 Word2Vec
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=32,
    window=3,
    min_count=1,
    workers=4,
    sg=1,
    seed=42
)

# 生成所有 Genres 的 embedding
all_genres = sorted({g for genres in sentences for g in genres})
genre_embeddings = {
    genre: w2v_model.wv[genre] for genre in all_genres
}

print(f"Genres 总数: {len(all_genres)}")
print("示例 embedding 维度:", genre_embeddings[all_genres[0]].shape)
print("示例 genre:", all_genres[0])
print("示例 embedding 前5维:", genre_embeddings[all_genres[0]][:5])


Genres 总数: 18
示例 embedding 维度: (32,)
示例 genre: Action
示例 embedding 前5维: [-0.00595829 -0.02383673  0.05213729 -0.02210927 -0.00703232]


In [16]:
import numpy as np

# 为每个电影生成 embedding：对其 Genres 的向量取平均
movie_embeddings = {}
for movie_id, genres in movies[["MovieID", "Genres"]].itertuples(index=False):
    genre_list = str(genres).split("|")
    vectors = [w2v_model.wv[g] for g in genre_list if g in w2v_model.wv]
    if vectors:
        movie_embeddings[movie_id] = np.mean(vectors, axis=0)
    else:
        movie_embeddings[movie_id] = np.zeros(w2v_model.vector_size, dtype=np.float32)

print(f"电影 embedding 数量: {len(movie_embeddings)}")
example_id = next(iter(movie_embeddings))
print("示例 MovieID:", example_id)
print("示例 embedding 前5维:", movie_embeddings[example_id][:5])


电影 embedding 数量: 3883
示例 MovieID: 1
示例 embedding 前5维: [-0.00501154 -0.0257538   0.04073036 -0.01369532  0.0357956 ]
